# Reef Imagery Pipeline — Jupyter Walkthrough

End-to-end run of the **Reef Benthic Visibility (BVI) pipeline** in Jupyter.

**Pipeline stages**:
1. Load environment (`.env` credentials, paths)
2. Pre-flight checks (input images, DGT CDD auth, CMEMS, IPMA)
3. Run the orchestrator: ACOLITE (or L2A fallback) → multi-scene Stumpf fusion → `run_predictor()` for BVI/SDB
4. Inspect outputs (BVI map, SDB map, drift reports)
5. Quick visualization

Compatible with the **DGT JupyterHub** at `dgt-jupyterhub.d.acnca.pt` — uses the same `.venv` and `.env` as the CLI run.

## 1. Environment setup

In [ ]:
import os, sys, json
from pathlib import Path

# Project root = parent of src/. Detect it relative to this notebook.
NB_DIR = Path.cwd()
PROJECT_DIR = NB_DIR if (NB_DIR / "src").exists() else NB_DIR.parent
if not (PROJECT_DIR / "src").exists():
    # Fall back: walk up to find src/orchestrator_run.py
    for p in [NB_DIR] + list(NB_DIR.parents):
        if (p / "src" / "orchestrator_run.py").exists():
            PROJECT_DIR = p
            break
print(f"Project dir: {PROJECT_DIR}")
sys.path.insert(0, str(PROJECT_DIR))
os.chdir(PROJECT_DIR)

In [ ]:
# Load .env (DGT_CDD_USERNAME, DGT_CDD_PASSWORD, CMEMS_*, etc.)
from dotenv import dotenv_values
env_path = PROJECT_DIR / ".env"
if env_path.exists():
    env = dotenv_values(env_path)
    for k, v in env.items():
        if v is not None and k not in os.environ:
            os.environ[k] = v
    print(f"Loaded {len(env)} env vars from {env_path}")
else:
    print(f"WARNING: {env_path} not found — DGT/CMEMS features may be disabled")
print(f"DGT_CDD_USERNAME: {'set' if os.environ.get('DGT_CDD_USERNAME') else 'NOT SET'}")
print(f"DGT_CDD_PASSWORD: {'set' if os.environ.get('DGT_CDD_PASSWORD') else 'NOT SET'}")
print(f"CMEMS_USER:       {'set' if os.environ.get('CMEMS_USER') else 'NOT SET'}")
print(f"PC_SDK_SUBSCRIPTION_KEY: {'set' if os.environ.get('PC_SDK_SUBSCRIPTION_KEY') else 'NOT SET'}")

In [ ]:
import logging
logging.basicConfig(level=logging.INFO,
                    format="%(asctime)s [%(levelname)s] %(message)s",
                    force=True)
log = logging.getLogger("reef_nb")

# Quick sanity checks
from src.constants import N_WATER, CLOUD_THRESHOLD, SNR_THRESHOLD, KD490_DEFAULT
print(f"N_WATER = {N_WATER}")
print(f"CLOUD_THRESHOLD = {CLOUD_THRESHOLD}")
print(f"SNR_THRESHOLD    = {SNR_THRESHOLD}")
print(f"KD490_DEFAULT    = {KD490_DEFAULT}")

## 2. Pre-flight: required input imagery

In [ ]:
from src.orchestrator_run import (
    IMAGE_A_B02, IMAGE_A_B03, IMAGE_B_B02, IMAGE_B_B03, OUTPUT_DIR,
    METADATA, TARGET_LAT, TARGET_LON
)

print(f"Target: ({TARGET_LAT}, {TARGET_LON})")
print(f"Output dir: {OUTPUT_DIR}")
print()
for label, p in [("A.B02", IMAGE_A_B02), ("A.B03", IMAGE_A_B03),
                 ("B.B02", IMAGE_B_B02), ("B.B03", IMAGE_B_B03)]:
    exists = p.exists()
    size_mb = p.stat().st_size / 1e6 if exists else 0
    print(f"  {label}: {'OK' if exists else 'MISSING'}  {p.name}  ({size_mb:.1f} MB)")
    print(f"           {p}")
print()
print("Metadata:")
for k, v in METADATA.items():
    print(f"  {k}: {v}")

In [ ]:
# Check external modules availability
from src.orchestrator_run import (
    HAS_CMEMS_KD, HAS_IPMA, HAS_IH_BATHY, HAS_DRIFT_MONITOR
)
print(f"CMEMS Kd490:    {'YES (live)' if HAS_CMEMS_KD else 'NO (static fallback)'}")
print(f"IPMA sea-state: {'YES' if HAS_IPMA else 'NO'}")
print(f"IH bathy:       {'YES' if HAS_IH_BATHY else 'NO'}")
print(f"Drift monitor:  {'YES' if HAS_DRIFT_MONITOR else 'NO'}")

# Check ACOLITE / SNAP availability (informational only)
from src.orchestrator_run import acolite_available, snap_gpt_available
print(f"ACOLITE:        {'YES' if acolite_available() else 'NO (L2A fallback)'}")
print(f"SNAP/gpt:       {'YES' if snap_gpt_available() else 'NO'}")

In [ ]:
import logging
logging.basicConfig(level=logging.INFO,
                    format="%(asctime)s [%(levelname)s] %(message)s",
                    force=True)
log = logging.getLogger("reef_nb")

# Quick sanity checks
from src.constants import N_WATER, CLOUD_THRESHOLD, SNR_THRESHOLD, KD490_DEFAULT
print(f"N_WATER = {N_WATER}")
print(f"CLOUD_THRESHOLD = {CLOUD_THRESHOLD}")
print(f"SNR_THRESHOLD    = {SNR_THRESHOLD}")
print(f"KD490_DEFAULT    = {KD490_DEFAULT}")

# If CMEMS live fetch is slow or unreachable on the JupyterHub,
# disable it for the rest of this session:
import os
if 'NO_CMEMS_LIVE' in os.environ:
    from src.cmems_kd490 import KD490_TABLE_LIVE as _T
    _T.clear()
    print("CMEMS live table disabled via NO_CMEMS_LIVE=1")

## 2. Pre-flight: required input imagery

In [ ]:
import time
t0 = time.time()
from src.orchestrator_run import main as orchestrator_main

DEPTH_M = 16.0  # target max-depth in metres
log.info("=== Launching orchestrator (depth=%.1f m) ===", DEPTH_M)

report = orchestrator_main(depth=DEPTH_M)

elapsed = time.time() - t0
log.info("Orchestrator finished in %.1f s", elapsed)
print()
print("=== Report keys ===")
print(list(report.keys()) if isinstance(report, dict) else type(report))

## 4. Inspect outputs

In [ ]:
import rasterio
import numpy as np

print(f"Output directory: {OUTPUT_DIR}")
if OUTPUT_DIR.exists():
    files = sorted(OUTPUT_DIR.iterdir())
    print(f"Files ({len(files)}):")
    for f in files:
        size_mb = f.stat().st_size / 1e6
        print(f"  {f.name:50s}  {size_mb:8.1f} MB")

In [ ]:
# Visualize the BVI map (image A) and confidence map
%matplotlib inline
import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import rasterio
from rasterio.plot import show

bvi_path = OUTPUT_DIR / "BVI_map_A_20250925.tif"
conf_path = OUTPUT_DIR / "Confidence_map_A_20250925.tif"
bvi_b_path = OUTPUT_DIR / "BVI_map_B_20231001.tif"

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
for ax, path, title in [(axes[0], bvi_path, "BVI (Image A — 2025-09-25)"),
                          (axes[1], bvi_b_path, "BVI (Image B — 2023-10-01)"),
                          (axes[2], conf_path, "Confidence (A)")]:
    if path and path.exists():
        with rasterio.open(path) as src:
            arr = src.read(1)
            arr_masked = np.ma.masked_invalid(arr)
            im = ax.imshow(arr_masked, cmap="turbo", vmin=0, vmax=1)
            ax.set_title(title)
            ax.set_xlabel(f"shape={arr.shape}, crs={src.crs}")
            plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    else:
        ax.set_title(f"{title}\n(MISSING)")
    ax.set_xticks([]); ax.set_yticks([])
fig.suptitle("Reef Benthic Visibility — Pipeline output",
             fontsize=14, weight="bold")
fig.tight_layout()
plt.show()

In [ ]:
# BVI histogram per scene
import numpy as np
import rasterio
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, path, label in [(axes[0], bvi_path, "A (2025-09-25)"),
                          (axes[1], bvi_b_path, "B (2023-10-01)")]:
    if path and path.exists():
        with rasterio.open(path) as src:
            arr = src.read(1).astype(np.float32)
        valid = arr[np.isfinite(arr) & (arr >= 0) & (arr <= 1)]
        if valid.size > 0:
            ax.hist(valid, bins=40, color="steelblue", edgecolor="white")
            ax.axvline(np.mean(valid), color="red", linestyle="--",
                        label=f"mean = {np.mean(valid):.3f}")
            ax.axvline(np.median(valid), color="darkgreen", linestyle=":",
                        label=f"median = {np.median(valid):.3f}")
        ax.set_xlabel("BVI")
        ax.set_ylabel("Pixel count")
        ax.set_title(f"BVI distribution — {label}")
        ax.legend()
        ax.set_xlim(0, 1)
fig.tight_layout()
plt.show()

## 5. Drift / quality reports (if enabled)

In [ ]:
from src.orchestrator_run import HAS_DRIFT_MONITOR
if HAS_DRIFT_MONITOR:
    from src.drift_history import export_history_json, export_history_csv
    from src.drift_report import export_html

    drift_json = OUTPUT_DIR / "drift_history.json"
    drift_csv  = OUTPUT_DIR / "drift_history.csv"
    drift_html = OUTPUT_DIR / "drift_report.html"

    for f in [drift_json, drift_csv, drift_html]:
        print(f"  {f.name}: {'OK' if f.exists() else 'NOT YET GENERATED'}")

    # If a HTML report exists, embed a link to open it
    if drift_html.exists():
        from IPython.display import display, FileLink
        display(FileLink(str(drift_html)))

## 6. Optional: re-run with a different depth or config

In [ ]:
import time
t0 = time.time()
from src.orchestrator_run import main as orchestrator_main

DEPTH_M = 16.0  # target max-depth in metres
log.info("=== Launching orchestrator (depth=%.1f m) ===", DEPTH_M)

report = orchestrator_main(depth=DEPTH_M)

elapsed = time.time() - t0
log.info("Orchestrator finished in %.1f s", elapsed)
print()
print("=== Report keys ===")
print(list(report.keys()) if isinstance(report, dict) else type(report))

### 3a. Tip: launch Jupyter from CLI in one command

If you can't reach the JupyterHub right now, you can launch the
notebook locally on the sandboxed environment with:

```bash
cd /Users/ssoares/Downloads/PI-PROJE/reef_imagery_pipeline
source .venv/bin/activate
jupyter lab notebooks/00_Reef_Pipeline_Jupyter.ipynb \
    --ServerApp.token='' --ServerApp.password='' \
    --ServerApp.allow_origin='*' --port 8888
```

## 7. Clean-up & shutdown

In [ ]:
# Nothing to clean up — files persist on the DGT JupyterHub storage.
# To re-run from scratch, delete the orchestrator report:
import os
report_path = OUTPUT_DIR / "orchestrator_report.json"
if report_path.exists():
    print(f"Latest report: {report_path} ({report_path.stat().st_size} bytes)")
print("Notebook complete.")